In [1]:
from pathlib import Path ## to use the Path class for file handling
from google import genai ## to use the Gemini API client
from datetime import datetime ## to get the current date and time
import json ## to handle JSON data
import pandas as pd ## to handle data in DataFrame format
import os ## to handle environment variables
import dtale

In [2]:
# Get current local date and time
now = datetime.now()

##extract the date in YYYY-MM-DD format
today = now.strftime("%Y-%m-%d")
print("\nToday's date: " + today)



Today's date: 2026-08-19


In [3]:
#configure the paths for the notes and prompt template files
notes_path = Path("data/import/sample_notes.md")
prompt_path = Path("data/import/extraction_prompt_template.md")

In [4]:
#storing the contents of the notes and prompt template files in variables
raw_notes = notes_path.read_text(encoding="utf-8")
prompt_template = prompt_path.read_text(encoding="utf-8")

print("Notes:")
print(raw_notes)

print("\nPrompt template:")
print(prompt_template)

Notes:
- ACE promotional layout meet 16th aug-6pm    
- ML pipeline debug 17th aug- YOLO- 8pm
- calisthenics routine this sunday
- shoot YouTube banter vid (this weekend)
- ask papa about schedule (today or tmmrw)
- flute practice 30 mins
- finish CCUS report by tomorrow please!!!!

Prompt template:
You are a task extraction engine for a personal scheduling system.

Today's date is: {today_date}
Timezone is: {timezone}

You will be given raw, unstructured notes—one item per line, written quickly
and informally. Extract each line into a structured JSON object.

For each item, output exactly these fields:
- "title": short, cleaned-up name of the task/event (string)
- "event_type": one of "lecture", "meeting", "routine", "evaluative", or "others"
- "start": object containing "dateTime", an ISO 8601 local datetime
  (YYYY-MM-DDTHH:MM), or null if no start date/time can be determined
- "end": object containing "dateTime", an ISO 8601 local datetime
  (YYYY-MM-DDTHH:MM), or null if no end/du

In [5]:
#configure the path for the Gemini API key file

gemini_api_key_path = Path("data/API_tokens_values/gemini_api_key.txt")

if not gemini_api_key_path.exists():
    raise FileNotFoundError(f"Gemini API key file not found: {gemini_api_key_path}")

with open(gemini_api_key_path, "r", encoding="utf-8") as f:
    gemini_api_key = f.read().strip()

#initialize the Gemini API client with the API key
client = genai.Client(api_key=gemini_api_key)

In [ ]:
#calling the Gemini API to generate content based on the prompt template and raw notes

#replace placeholders in the prompt template with actual values
prompt = (
    prompt_template
    .replace("{today_date}", today)
    .replace("{timezone}", "Asia/Kolkata")
    .replace("{notes}", raw_notes)
)

response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents=prompt,
)

print(response.text)

[
  {
    "title": "ACE promotional layout meeting",
    "event_type": "meeting",
    "start": {
      "dateTime": "2026-08-16T18:00"
    },
    "end": {
      "dateTime": null
    },
    "estimated_duration": null,
    "importance": 3,
    "urgency": 2
  },
  {
    "title": "ML pipeline debug - YOLO",
    "event_type": "others",
    "start": {
      "dateTime": "2026-08-17T20:00"
    },
    "end": {
      "dateTime": null
    },
    "estimated_duration": null,
    "importance": 3,
    "urgency": 2
  },
  {
    "title": "Calisthenics routine",
    "event_type": "routine",
    "start": {
      "dateTime": "2026-08-23T00:00"
    },
    "end": {
      "dateTime": null
    },
    "estimated_duration": null,
    "importance": 3,
    "urgency": 3
  },
  {
    "title": "Shoot YouTube banter video",
    "event_type": "others",
    "start": {
      "dateTime": "2026-08-22T00:00"
    },
    "end": {
      "dateTime": null
    },
    "estimated_duration": null,
    "importance": 2,
    "urgency":

In [7]:
#storing the response from the Gemini API in a JSON file

response_text = response.text.strip()

# If Gemini returns pure JSON text
data = json.loads(response_text)

output_path = Path("data/export/response.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {output_path}")

Saved response to data\export\response.json


In [8]:
with open("data/export/response.json", "r", encoding="utf-8") as f:
    tasks = json.load(f)

df = pd.DataFrame(tasks)

# Flatten nested "start" and "end" objects into DataFrame columns.
df = pd.json_normalize(tasks)

schema_columns = [
    "title",
    "event_type",
    "start.dateTime",
    "end.dateTime",
    "estimated_duration",
    "importance",
    "urgency",
]

df = df.reindex(columns=schema_columns)

df

,title,event_type,start.dateTime,end.dateTime,estimated_duration,importance,urgency
0,ACE promotional layout meeting,meeting,2026-08-16T18:00,NaN,NaN,3,2
1,ML pipeline debug - YOLO,others,2026-08-17T20:00,NaN,NaN,3,2
2,Calisthenics routine,routine,2026-08-23T00:00,NaN,NaN,3,3
3,Shoot YouTube banter video,others,2026-08-22T00:00,NaN,NaN,2,3
4,Ask papa about schedule,others,2026-08-19T00:00,NaN,NaN,3,5
5,Flute practice,routine,NaN,NaN,30.0,3,2
6,Finish CCUS report,evaluative,NaN,2026-08-20T23:59,NaN,4,4


In [9]:
# Open D-Tale editor
d = dtale.show(
    df,
    name="Scheduler task review",
    allow_cell_edits=True,
)

d.open_browser()

In [11]:
edited_df = d.data

# 1. Convert float column to nullable integer, keeping nulls as NaN
edited_df['estimated_duration'] = edited_df['estimated_duration'].astype('Int64')
edited_df['start.dateTime'] = pd.to_datetime(edited_df['start.dateTime'])
edited_df['end.dateTime'] = pd.to_datetime(edited_df['end.dateTime'])



edited_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   title               7 non-null      str           
 1   event_type          7 non-null      str           
 2   start.dateTime      5 non-null      datetime64[us]
 3   end.dateTime        1 non-null      datetime64[us]
 4   estimated_duration  1 non-null      Int64         
 5   importance          7 non-null      int64         
 6   urgency             7 non-null      int64         
dtypes: Int64(1), datetime64[us](2), int64(2), str(2)
memory usage: 531.0 bytes


In [13]:
edited_df

,title,event_type,start.dateTime,end.dateTime,estimated_duration,importance,urgency
0,ACE promotional layout meeting,meeting,2026-08-16 18:00:00,NaT,<NA>,3,2
1,ML pipeline debug - YOLO,others,2026-08-17 20:00:00,NaT,<NA>,3,2
2,Calisthenics routine,routine,2026-08-23 00:00:00,NaT,<NA>,3,3
3,Shoot YouTube banter video,others,2026-08-22 00:00:00,NaT,<NA>,2,3
4,Ask papa about schedule,others,2026-08-19 00:00:00,NaT,<NA>,3,5
5,Flute practice,routine,NaT,NaT,30,3,2
6,Finish CCUS report,evaluative,NaT,2026-08-20 23:59:00,<NA>,4,4


In [14]:
#export the edited DataFrame to a CSV file
output_csv_path = Path("data/export/CSV_2.csv")
edited_df.to_csv(output_csv_path, index=False)